### Cell 1 ? Imports and setup

Imports PyTorch and supporting utilities, sets reproducible random seeds, and selects the available CPU or GPU device.


In [11]:
import re
import random
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


### Cell 2 ? Sentence-pair dataset

Defines sentence pairs labeled as consecutive or unrelated, then shuffles and splits them into training and test examples.


In [12]:
sentence_pairs = [
    ("the cat is small", "it likes milk", 1),
    ("the dog is happy", "it plays outside", 1),
    ("birds can fly", "they have wings", 1),
    ("the sun is bright", "the sky is blue", 1),
    ("fish live in water", "they can swim", 1),
    ("the baby is sleeping", "the room is quiet", 1),
    ("the cat is small", "cars need fuel", 0),
    ("birds can fly", "the baby is sleeping", 0),
    ("fish live in water", "the sun is bright", 0),
    ("the dog is happy", "they have wings", 0),
    ("cars need fuel", "it likes milk", 0),
    ("the sky is blue", "fish live in water", 0),
]

# Use a few examples for training and keep a few for a simple check
random.shuffle(sentence_pairs)
train_pairs = sentence_pairs[:10]
test_pairs = sentence_pairs[10:]
print("Training examples:", len(train_pairs))
print("Test examples:", len(test_pairs))

Training examples: 10
Test examples: 2


### Cell 3 ? Vocabulary

Defines BERT special tokens, tokenizes the text, builds the vocabulary, and creates mappings between words and token IDs.


In [13]:
PAD, UNK, CLS, SEP, MASK = "[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"
special_tokens = [PAD, UNK, CLS, SEP, MASK]

def tokenize(text):
    return re.findall(r"[a-z]+", text.lower())

all_words = []
for first, second, _ in sentence_pairs:
    all_words += tokenize(first) + tokenize(second)

vocab_words = sorted(set(all_words))
vocab = {word: i for i, word in enumerate(special_tokens + vocab_words)}
id_to_word = {i: word for word, i in vocab.items()}

PAD_ID = vocab[PAD]
UNK_ID = vocab[UNK]
CLS_ID = vocab[CLS]
SEP_ID = vocab[SEP]
MASK_ID = vocab[MASK]
VOCAB_SIZE = len(vocab)
print("Vocabulary size:", VOCAB_SIZE)
print(vocab)

Vocabulary size: 38
{'[PAD]': 0, '[UNK]': 1, '[CLS]': 2, '[SEP]': 3, '[MASK]': 4, 'baby': 5, 'birds': 6, 'blue': 7, 'bright': 8, 'can': 9, 'cars': 10, 'cat': 11, 'dog': 12, 'fish': 13, 'fly': 14, 'fuel': 15, 'happy': 16, 'have': 17, 'in': 18, 'is': 19, 'it': 20, 'likes': 21, 'live': 22, 'milk': 23, 'need': 24, 'outside': 25, 'plays': 26, 'quiet': 27, 'room': 28, 'sky': 29, 'sleeping': 30, 'small': 31, 'sun': 32, 'swim': 33, 'the': 34, 'they': 35, 'water': 36, 'wings': 37}


### Cell 4 ? Dataset and batches

Builds fixed-length model inputs with sentence segments, a randomly masked token, padding, and attention masks; then creates PyTorch datasets and data loaders.


In [ ]:
MAX_LEN=14

def make_example(sentence_a,sentence_b,is_next):
    words_a=tokenize(sentence_a)
    words_b=tokenize(sentence_b)
    #Combine the sentences using special tokens
    tokens=[CLS]+words_a+[SEP]+words_b+[SEP]
    #0 IDENTIFIES SENTENCE A 1 IDENTIFIES SENTENCE B
    segments=[0]*(len(words_a)+2) + [1]*(len(words_b)+1)
    #if words present return its wordsid else return unkid
    ids=[vocab.get(word,UNK_ID) for word in tokens]

    # Labels are ignored (-100) except at the masked word position BERT to predict the original word only at positions you selected for masking.
    #nn.CrossEntropyLoss(), the default ignore_index is -100. It tells the loss function to ignore those positions
    mlm_labels=[-100]*len(ids)
    #skips the first and last positions.excludes the separator and classification tokens
    possible_positions=[i for i in range(1,len(ids)-1)if ids[i] not in (CLS_ID,SEP_ID)]
    if possible_positions:
        #random make a id change 6 to 4 love to my
        position = random.choice(possible_positions)
        #bert see mask and redict 6 which is origional id
        mlm_labels[position] = ids[position]
        ids[position] = MASK_ID

    #This limits each list to 14 positions.
    ids = ids[:MAX_LEN]
    #14 segments
    segments = segments[:MAX_LEN]
    #14 labels
    mlm_labels = mlm_labels[:MAX_LEN]
    #1 meansreal id 0 means padding
    attention = [1] * len(ids)

    #if 14 - 11 =3
    padding = MAX_LEN - len(ids)
    #Fill unused positions
    ids += [PAD_ID] * padding
    #Give padding a default segment
    segments += [0] * padding
    #Ignore padding in MLM loss
    mlm_labels += [-100] * padding
    #Tell attention to ignore padding
    attention += [0] * padding
    #Return PyTorch tensors
    return (torch.tensor(ids), torch.tensor(segments),
            torch.tensor(attention), torch.tensor(mlm_labels), torch.tensor(is_next))

class TinyBERTDataset(Dataset):
    #prepare the examples
    def __init__(self, pairs):
        self.examples = [make_example(a, b, label) for a, b, label in pairs]
    #returns the number of examples.
    def __len__(self):
        return len(self.examples)
    #returns one example by its index.
    def __getitem__(self, index):
        return self.examples[index]
#Create the DataLoaders
train_loader = DataLoader(TinyBERTDataset(train_pairs), batch_size=2, shuffle=True)
test_loader = DataLoader(TinyBERTDataset(test_pairs), batch_size=2)
#Inspect the batch shapes
batch = next(iter(train_loader))
for item in batch:
    print(item.shape)

torch.Size([2, 14])
torch.Size([2, 14])
torch.Size([2, 14])
torch.Size([2, 14])
torch.Size([2])


### Cell 5 ? BabyBERT model

Defines the small Transformer model with word, position, and segment embeddings plus masked-language-modeling and next-sentence-prediction output heads.


In [ ]:
#Define the model class nn.Module, your model can register its layers vocab_size Number of available tokens
class BabyBERT(nn.Module):
    def __init__(self, vocab_size, max_len, embed_size=32, heads=4, layers=2):
        #your vocabulary converts words into IDs. The embedding layer converts those IDs into vectors of 32 numbers.
        super().__init__()
        #vocab_size to 32 each vecotor
        self.word_embedding = nn.Embedding(vocab_size, embed_size, padding_idx=PAD_ID)
        #Position embeddings give each position its own learned vector.
        self.position_embedding = nn.Embedding(max_len, embed_size)
        #You have two sentences, so you use two segment IDs:0sentencea 1 sentenceb
        self.segment_embedding = nn.Embedding(2, embed_size)
        #WORD + POSITION + SEGMENT
        #Create the Transformer encoder layer Self-attention + feed-forward network embed_size=32, heads=4, Q, K, and V? are inside the multi-head attention mechanism used by nn.TransformerEncoderLayer.
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_size, nhead=heads, dim_feedforward=64,
            dropout=0.1, batch_first=True
        )
        #SECONDS LAYER Learns more contextual relationships
        #encoderlayer>2then 
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=layers)
        #for every token position, this layer produces one score for every word in your vocabulary.if vocab_size = 1000, the model produces 1,000 scores for each position.
        self.mlm_head = nn.Linear(embed_size, vocab_size)
        #nsp produces 2 score 0 or 1
        self.nsp_head = nn.Linear(embed_size, 2)
    #This method defines what happens when you pass data through your model.
        # The model receives three inputs:
        #input_ids: token IDs.
        #segment_ids: sentence membership IDs.
        #attention_mask: identifies real tokens versus padding.
    def forward(self, input_ids, segment_ids, attention_mask):
        batch_size, seq_len = input_ids.shape
        #Create position IDs
        positions = torch.arange(seq_len, device=input_ids.device).unsqueeze(0).expand(batch_size, seq_len)
        #Add the embeddings If each embedding has shape [2, 14, 32], the combined tensor x also has shape [2, 14, 32].
        x = (self.word_embedding(input_ids) +
             self.position_embedding(positions) +
             self.segment_embedding(segment_ids))
        #1 for real tokens.0 for padding.
        padding_mask = attention_mask == 0
        #encoder uses this mask to prevent padded positions from being used as keys in attention. This reduces the influence of padding on real tokens.
        encoded = self.encoder(x, src_key_padding_mask=padding_mask)
        #Make MLM predictions
        #The MLM head transforms each 32-number vector into vocab_size scores.
        mlm_logits = self.mlm_head(encoded)
        # : selects every example in the batch.0 selects position zero, where [CLS] is located.: selects all 32 embedding values.
        nsp_logits = self.nsp_head(encoded[:, 0, :])
        #Return both outputs
        return mlm_logits, nsp_logits
#Create the model vocabsize how many tokens ids exist maxlen 14
model = BabyBERT(VOCAB_SIZE, MAX_LEN).to(device)
print(model)

BabyBERT(
  (word_embedding): Embedding(38, 32, padding_idx=0)
  (position_embedding): Embedding(14, 32)
  (segment_embedding): Embedding(2, 32)
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=32, out_features=32, bias=True)
        )
        (linear1): Linear(in_features=32, out_features=64, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=64, out_features=32, bias=True)
        (norm1): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (mlm_head): Linear(in_features=32, out_features=38, bias=True)
  (nsp_head): Linear(in_features=32, out_features=2, bias=True)
)


# BERT Training Code — Simple Summary in Markdown

## 1. What does this code do?

This code trains our small BERT model to perform two tasks:

- MLM: Predict the original word hidden by `[MASK]`.
- NSP: Predict whether Sentence B follows Sentence A.

It repeats training for 20 epochs, trying to improve the model's predictions by updating its weights.

## 2. Training process

Input batch from `train_loader`

BERT predicts masked words and sentence relationships

MLM Loss

Word prediction error

NSP Loss

Sentence prediction error

Add losses → Backpropagation → Update weights

Repeat for 20 epochs

## 3. Important code explained

| Code                                     | Simple meaning                                                                             |
| ---------------------------------------- | ------------------------------------------------------------------------------------------ |
| `nn.CrossEntropyLoss(ignore_index=-100)` | Calculates MLM prediction error while ignoring positions that are not masked-word targets. |
| `nn.CrossEntropyLoss()`                  | Calculates NSP classification error.                                                       |
| `Adam(model.parameters(), lr=0.001)`     | Updates the model's weights using the Adam optimizer.                                      |
| `EPOCHS = 20`                            | Trains over the dataset 20 times.                                                          |
| `model.train()`                          | Enables training behavior, including dropout.                                              |
| `optimizer.zero_grad()`                  | Clears gradients from the previous batch.                                                  |
| `model(...)`                             | Produces MLM and NSP predictions.                                                          |
| `loss.backward()`                        | Calculates gradients to show how weights should change.                                    |
| `optimizer.step()`                       | Updates the model's weights.                                                               |
| `total_loss += loss.item()`              | Adds the current batch's loss to the running total.                                        |

## 4. Why reshape the MLM predictions?

```
loss_mlm = mlm_loss_fn(    mlm_logits.reshape(-1, VOCAB_SIZE),    mlm_labels.reshape(-1))
```

The model predicts a word for every sequence position, but PyTorch's cross-entropy loss expects a two-dimensional table of class scores.

For example, if the batch has 2 sequences, each with 14 tokens:

- MLM predictions: `[2, 14, VOCAB_SIZE]`
- Reshaped predictions: `[28, VOCAB_SIZE]`
- Reshaped labels: `[28]`

Each of the 28 positions has a target label, but positions labeled `-100` are ignored.

## 5. How are the losses combined?

```
loss = loss_mlm + loss_nsp
```

The model learns from both tasks together. The combined loss is used to calculate gradients and update the same model weights.


### Cell 6 ? Model training

Trains BabyBERT for 20 epochs using combined masked-token and next-sentence losses, updating its weights with Adam.


In [ ]:
#MLM: Learn to predict the original word hidden by [MASK].
mlm_loss_fn = nn.CrossEntropyLoss(ignore_index=-100)
#NSP: Learn to predict whether Sentence B follows Sentence A.
nsp_loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


EPOCHS = 20
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0

    for input_ids, segment_ids, attention_mask, mlm_labels, nsp_labels in train_loader:
        input_ids = input_ids.to(device)
        segment_ids = segment_ids.to(device)
        attention_mask = attention_mask.to(device)
        mlm_labels = mlm_labels.to(device)
        nsp_labels = nsp_labels.to(device)

        optimizer.zero_grad()
        mlm_logits, nsp_logits = model(input_ids, segment_ids, attention_mask)

        loss_mlm = mlm_loss_fn(mlm_logits.reshape(-1, VOCAB_SIZE), mlm_labels.reshape(-1))
        loss_nsp = nsp_loss_fn(nsp_logits, nsp_labels)
        loss = loss_mlm + loss_nsp

        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1:02d} | loss: {total_loss / len(train_loader):.4f}")

Epoch 01 | loss: 4.5819
Epoch 02 | loss: 4.1283
Epoch 03 | loss: 3.8236
Epoch 04 | loss: 3.5722
Epoch 05 | loss: 3.3025
Epoch 06 | loss: 3.2032
Epoch 07 | loss: 2.9948
Epoch 08 | loss: 2.7727
Epoch 09 | loss: 2.6698
Epoch 10 | loss: 2.4622
Epoch 11 | loss: 2.4431
Epoch 12 | loss: 2.1658
Epoch 13 | loss: 2.0944
Epoch 14 | loss: 1.9602
Epoch 15 | loss: 1.8019
Epoch 16 | loss: 1.6914
Epoch 17 | loss: 1.5489
Epoch 18 | loss: 1.4044
Epoch 19 | loss: 1.1925
Epoch 20 | loss: 1.1017


# BERT NSP Prediction — Simple Summary

This code uses our trained BERT model to predict whether Sentence B follows Sentence A.

### How it works

1. `model.eval()` — Switches the model to prediction mode.
2. `tokenize()` — Splits sentences into words.
3. `[CLS]` and `[SEP]` — Combines both sentences in BERT's required format.
4. Segment IDs — Mark Sentence A with `0` and Sentence B with `1`.
5. Vocabulary — Converts words into numbers.
6. Padding and attention mask — Makes the input the correct length and identifies real tokens.
7. `torch.no_grad()` — Makes predictions without calculating gradients.
8. `model(...)` — Gets the model's NSP prediction scores.
9. `argmax()` — Chooses the class with the highest score.
10. `return` — Displays the predicted result.

### Simple flow

Two sentences

Convert words into numbers

Send input to trained BERT

Predict sentence relationship

Likely follows / Likely does not follow

### Remember

- Training: Teaches the model by updating its weights.
- Prediction: Uses the trained model to make an answer without updating its weights.
- NSP: Predicts whether the second sentence follows the first according to what the model learned.

The model is small, so its predictions may not always be correct.

### Cell 7 ? Next-sentence prediction

Prepares two sentences as model inputs and predicts whether the second sentence is likely to follow the first.


In [17]:
def predict_nsp(sentence_a, sentence_b):
    model.eval()
    # No masked word is needed for NSP prediction
    words_a, words_b = tokenize(sentence_a), tokenize(sentence_b)
    tokens = [CLS] + words_a + [SEP] + words_b + [SEP]
    segments = [0] * (len(words_a) + 2) + [1] * (len(words_b) + 1)
    ids = [vocab.get(word, UNK_ID) for word in tokens][:MAX_LEN]
    segments = segments[:MAX_LEN]
    mask = [1] * len(ids)
    pad = MAX_LEN - len(ids)
    ids += [PAD_ID] * pad
    segments += [0] * pad
    mask += [0] * pad

    with torch.no_grad():
        _, logits = model(torch.tensor([ids], device=device),
                          torch.tensor([segments], device=device),
                          torch.tensor([mask], device=device))
        prediction = logits.argmax(dim=1).item()
    return "Likely follows" if prediction == 1 else "Likely does not follow"

print(predict_nsp("the cat is small", "it likes milk"))
print(predict_nsp("the cat is small", "cars need fuel"))

Likely follows
Likely does not follow


# BERT MLM Prediction — Simple Summary

This code uses our trained BERT model to predict a missing word represented by `[MASK]`.

Example: `birds [MASK] fly they have wings`

### How it works

1. Split the sentence: Separate the text around `[MASK]`.
2. Tokenize: Convert the sentence into individual words while keeping `[MASK]`.
3. Convert words to IDs: Change each word into a number using the vocabulary.
4. Find the mask: Find the position of `[MASK]` in the input.
5. Prepare inputs: Create segment IDs and an attention mask.
6. Predict: BERT calculates scores for possible words at the masked position.
7. Choose the word: `argmax()` selects the word with the highest score.
8. Return the word: `id_to_word` converts the predicted number back into a word.

### Simple flow

`birds [MASK] fly they have wings`

Convert words into IDs

BERT predicts the missing word

Return the predicted word

### Remember

- MLM means Masked Language Modeling.
- `[MASK]` represents the missing word.
- `mlm_logits[0, mask_position]` gets the scores for possible words at the masked position.
- `argmax()` selects the highest-scoring word.
- `id_to_word` converts the predicted ID back to text.

Important: The predicted word depends on what the small model learned during training, so it may not always be correct.

### Cell 8 ? Mask-aware tokenizer

Defines a tokenizer that keeps the `[MASK]` token intact while splitting the rest of the text into words and punctuation.


In [26]:
import re

def simple_tokenizer(text):
    # Keep words and special tokens like [MASK] together
    return re.findall(r"\[MASK\]|\w+|[^\w\s]", text.lower())

### Cell 9 ? Masked-word prediction

Prepares text containing `[MASK]`, asks the trained model to predict the missing token, and converts the predicted ID back to a word.


In [43]:
def predict_masked_word(sentence):
    # Split the sentence around [MASK]
    parts = sentence.lower().split("[mask]")

    # Convert words into tokens, preserving [MASK]
    tokens = []
    for i, part in enumerate(parts):
        tokens.extend(simple_tokenizer(part))
        if i < len(parts) - 1:
            tokens.append("[MASK]")

    # Convert tokens into IDs
    ids = [
        MASK_ID if word == "[MASK]"
        else vocab.get(word, UNK_ID)
        for word in tokens
    ][:MAX_LEN]

    # Find the MASK position
    if MASK_ID not in ids:
        raise ValueError("MASK token is missing from the vocabulary.")

    mask_position = ids.index(MASK_ID)

    # Prepare model inputs
    segments = [0] * len(ids)
    attention_mask = [1] * len(ids)

    input_ids = torch.tensor([ids], dtype=torch.long)
    token_type_ids = torch.tensor([segments], dtype=torch.long)
    attention_mask = torch.tensor([attention_mask], dtype=torch.long)

    # Predict the missing word
    model.eval()
    with torch.no_grad():
        mlm_logits, _ = model(input_ids, token_type_ids, attention_mask)
        predicted_id = mlm_logits[0, mask_position].argmax().item()

    return id_to_word[predicted_id]


print("Predicted word:", predict_masked_word("birds [MASK] fly they have wings"))

Predicted word: the
